# 02 — Modélisation

Ce notebook compare plusieurs modèles de régression pour prédire le prix des véhicules :

- DummyRegressor (stratégies moyenne et médiane) comme référence minimale ;
- LinearRegression ;
- Ridge, RidgeCV, Lasso, ElasticNet et ElasticNetCV (modèles linéaires régularisés) ;
- DecisionTreeRegressor ;
- RandomForestRegressor ;
- XGBoost (XGBRegressor).

Les modèles sont entraînés sur la même séparation entraînement/test (80 % / 20 %, `random_state=42`) et évalués avec les mêmes métriques (MAE, RMSE, R²). Une validation croisée à 5 plis est utilisée pour les principaux modèles.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/interim/vehicles_clean.csv")

df.head()

,YEAR,KM,BODYDOORS,PASSENGERS,MAKE,MODEL,BODYDESCF,DRIVETRAIN,FUEL,TRANSDESCF,COLORF,PRICE
0,2022,73672,4,5,Hyundai,Kona,VUS,4 roues motrices,Essence,Automatique,Noir,20995
1,2023,47396,4,5,Buick,Envision,VUS,4 roues motrices,Essence,Automatique,Blanc,27995
2,2021,54507,4,5,Nissan,Versa,Berline,Autre,Essence,Automatique,Charbon,16495
3,2022,82689,4,5,Nissan,LEAF,Hatchback,Autre,Électrique,Automatique,Noir,19495
4,2022,95055,4,5,Volkswagen,Tiguan,VUS,4 roues motrices,Essence,Automatique,Gris,23495


In [3]:
df.shape

(1748, 12)

## Séparation des variables explicatives et de la cible

In [4]:
X = df.drop(columns="PRICE")
y = df["PRICE"]

In [5]:
X.shape, y.shape

((1748, 11), (1748,))

## Séparation des données d'entraînement et de test

Les données sont séparées en deux groupes :

- 80 % pour entraîner les modèles ;
- 20 % pour évaluer leurs performances sur des données non utilisées pendant l'entraînement.

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (1398, 11)
X_test  : (350, 11)
y_train : (1398,)
y_test  : (350,)


## Préparation des variables

Les variables numériques sont conservées telles quelles.

Les variables catégorielles sont transformées avec `OneHotEncoder` afin de convertir les catégories en valeurs numériques utilisables par les modèles.

In [9]:
variables_numeriques = [
    "YEAR",
    "KM",
    "BODYDOORS",
    "PASSENGERS"
]

variables_categorielles = [
    "MAKE",
    "MODEL",
    "BODYDESCF",
    "DRIVETRAIN",
    "FUEL",
    "TRANSDESCF",
    "COLORF"
]

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            "passthrough",
            variables_numeriques
        )
    ]
)

In [12]:
X_train_prepared = preprocessor.fit_transform(X_train)

X_test_prepared = preprocessor.transform(X_test)

In [13]:
print("Avant préparation :", X_train.shape)
print("Après préparation :", X_train_prepared.shape)

print("Test avant :", X_test.shape)
print("Test après :", X_test_prepared.shape)

Avant préparation : (1398, 11)
Après préparation : (1398, 287)
Test avant : (350, 11)
Test après : (350, 287)


## Modèle 1 — DummyRegressor

Le `DummyRegressor` sert de modèle de référence.

Il effectue une prédiction très simple sans réellement apprendre les relations entre les variables.

Il permet de vérifier si les modèles plus avancés font réellement mieux qu'une stratégie naïve.

In [14]:
from sklearn.dummy import DummyRegressor

In [15]:
dummy_model = DummyRegressor(strategy="mean")

In [16]:
dummy_model.fit(X_train_prepared, y_train)

,strategy,'mean'
,constant,None
,quantile,None


In [17]:
y_pred_dummy = dummy_model.predict(X_test_prepared)

In [18]:
print("Prix moyen appris :", y_train.mean())

print("Premières prédictions :")
print(y_pred_dummy[:5])

Prix moyen appris : 23801.15522174535
Premières prédictions :
[23801.15522175 23801.15522175 23801.15522175 23801.15522175
 23801.15522175]


## Évaluation du DummyRegressor

Les performances sont évaluées avec trois métriques :

- **MAE** : erreur moyenne entre le prix réel et le prix prédit.
- **RMSE** : similaire à la MAE, mais pénalise davantage les grosses erreurs.
- **R²** : mesure la capacité du modèle à expliquer les variations du prix.

In [19]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

In [20]:
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
rmse_dummy = root_mean_squared_error(y_test, y_pred_dummy)
r2_dummy = r2_score(y_test, y_pred_dummy)

print("MAE  :", mae_dummy)
print("RMSE :", rmse_dummy)
print("R²   :", r2_dummy)

MAE  : 5740.729961169018
RMSE : 7872.426505752306
R²   : -0.016472071090171125


### Interprétation

Le DummyRegressor obtient une MAE d'environ 5 741 $, un RMSE d'environ 7 872 $ et un R² proche de 0 (-0,016).

Le R² est légèrement négatif parce que la moyenne apprise sur l'entraînement n'est pas exactement la moyenne du jeu de test.

Ce modèle sert de référence minimale. Les modèles suivants devront obtenir une erreur plus faible et un R² plus élevé pour être considérés comme plus performants.

## Modèle 2 — LinearRegression

La régression linéaire cherche une relation globale entre les caractéristiques du véhicule et son prix.

Elle sert de premier véritable modèle prédictif.

In [21]:
from sklearn.linear_model import LinearRegression

In [22]:
linear_model = LinearRegression()

linear_model.fit(X_train_prepared, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [23]:
y_pred_linear = linear_model.predict(X_test_prepared)

In [24]:
mae_linear = mean_absolute_error(y_test, y_pred_linear)
rmse_linear = root_mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

print("MAE  :", mae_linear)
print("RMSE :", rmse_linear)
print("R²   :", r2_linear)

MAE  : 2274.1896321474933
RMSE : 4242.584890067344
R²   : 0.7047844005183247


### Interprétation

La régression linéaire améliore nettement les résultats par rapport au DummyRegressor.

- MAE : environ 2 274 $
- RMSE : environ 4 243 $
- R² : environ 0,70

Le modèle parvient donc à expliquer une grande partie des variations de prix à partir des caractéristiques des véhicules.

## Modèle 3 — RandomForestRegressor

Le RandomForestRegressor combine plusieurs arbres de décision.

Contrairement à la régression linéaire, il peut apprendre des relations plus complexes entre les caractéristiques du véhicule et son prix.

In [25]:
from sklearn.ensemble import RandomForestRegressor

On construit 100 arbres et on combine leurs prédictions.

In [26]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [27]:
random_forest_model.fit(X_train_prepared, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [28]:
y_pred_forest = random_forest_model.predict(X_test_prepared)

In [29]:
mae_forest = mean_absolute_error(y_test, y_pred_forest)
rmse_forest = root_mean_squared_error(y_test, y_pred_forest)
r2_forest = r2_score(y_test, y_pred_forest)

print("MAE  :", mae_forest)
print("RMSE :", rmse_forest)
print("R²   :", r2_forest)

MAE  : 1972.8234285714284
RMSE : 4537.212542784391
R²   : 0.6623579966037504


### Interprétation du RandomForestRegressor

Le RandomForestRegressor améliore nettement les résultats par rapport au DummyRegressor.

Avec les paramètres actuels, il obtient une MAE plus faible que la régression linéaire (environ 1 973 $ contre 2 274 $), mais un RMSE plus élevé (environ 4 537 $ contre 4 243 $) et un R² plus faible (environ 0,66 contre 0,70).

Il commet donc des erreurs plus faibles en moyenne, mais certaines erreurs importantes pénalisent davantage le RMSE et le R².

Cela montre qu'un modèle plus complexe n'est pas nécessairement plus performant pour un jeu de données donné.

## Comparaison des modèles

In [30]:
resultats = pd.DataFrame({
    "Modèle": [
        "DummyRegressor",
        "LinearRegression",
        "RandomForestRegressor"
    ],
    "MAE": [
        mae_dummy,
        mae_linear,
        mae_forest
    ],
    "RMSE": [
        rmse_dummy,
        rmse_linear,
        rmse_forest
    ],
    "R²": [
        r2_dummy,
        r2_linear,
        r2_forest
    ]
})

resultats

,Modèle,MAE,RMSE,R²
0,DummyRegressor,5740.729961,7872.426506,-0.016472
1,LinearRegression,2274.189632,4242.584890,0.704784
2,RandomForestRegressor,1972.823429,4537.212543,0.662358


## Validation croisée

Une validation croisée à 5 plis est utilisée pour vérifier si les performances des modèles restent stables sur différentes séparations des données.

In [31]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

In [32]:
pipeline_linear = Pipeline([
    ("preparation", preprocessor),
    ("modele", LinearRegression())
])

In [33]:
scores_linear = cross_val_score(
    pipeline_linear,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_linear)
print("R² moyen   :", scores_linear.mean())

R² par pli : [0.51853748 0.72885534 0.81956072 0.81975752 0.72696853]
R² moyen   : 0.7227359198427625


### Interprétation de la validation croisée

La régression linéaire obtient un R² moyen d'environ 0,72 sur les 5 plis.

Les résultats varient toutefois fortement selon les plis, d'environ 0,52 à 0,82. Le premier pli est nettement plus difficile que les autres.

La moyenne obtenue est proche du R² observé sur le jeu de test (environ 0,70), mais cette variabilité montre qu'une seule séparation train/test ne suffit pas pour juger de la stabilité d'un modèle.

In [34]:
pipeline_forest = Pipeline([
    ("preparation", preprocessor),
    ("modele", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [35]:
scores_forest = cross_val_score(
    pipeline_forest,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_forest)
print("R² moyen   :", scores_forest.mean())

R² par pli : [0.58412713 0.72210747 0.80719135 0.83544785 0.60587132]
R² moyen   : 0.7109490254741745


### Interprétation de la validation croisée du RandomForestRegressor

Le RandomForestRegressor obtient un R² moyen d'environ 0,71 sur les 5 plis.

Les performances sont globalement bonnes, mais légèrement inférieures à celles de la régression linéaire, qui obtient un R² moyen d'environ 0,72.

Pour ce jeu de données et avec les paramètres actuels, le RandomForestRegressor n'apporte donc pas d'amélioration du R² moyen en validation croisée par rapport à la régression linéaire.

# Expérimentation des variantes

## Variante du DummyRegressor

Une deuxième stratégie naïve est testée en utilisant la médiane des prix au lieu de la moyenne.

In [36]:
dummy_median = DummyRegressor(strategy="median")

dummy_median.fit(X_train_prepared, y_train)

y_pred_dummy_median = dummy_median.predict(X_test_prepared)

In [37]:
mae_dummy_median = mean_absolute_error(y_test, y_pred_dummy_median)
rmse_dummy_median = root_mean_squared_error(y_test, y_pred_dummy_median)
r2_dummy_median = r2_score(y_test, y_pred_dummy_median)

print("MAE  :", mae_dummy_median)
print("RMSE :", rmse_dummy_median)
print("R²   :", r2_dummy_median)

MAE  : 5543.428571428572
RMSE : 7814.294593883698
R²   : -0.001515742359488037


### Comparaison des stratégies Dummy

La stratégie utilisant la médiane obtient des résultats légèrement meilleurs que celle utilisant la moyenne.

Cependant, les deux modèles restent des références naïves avec un R² proche de 0.

## Ridge

Ridge est une régression linéaire avec régularisation.

La régularisation réduit l'importance excessive de certains coefficients afin de limiter le surapprentissage.

Deux valeurs de `alpha` seront testées :

- `alpha = 1` : régularisation modérée ;
- `alpha = 10` : régularisation plus forte.

In [38]:
from sklearn.preprocessing import StandardScaler

preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            StandardScaler(),
            variables_numeriques
        )
    ]
)

In [39]:
from sklearn.linear_model import Ridge

In [40]:
ridge_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=1))
])

ridge_1.fit(X_train, y_train)

y_pred_ridge_1 = ridge_1.predict(X_test)

In [41]:
mae_ridge_1 = mean_absolute_error(y_test, y_pred_ridge_1)
rmse_ridge_1 = root_mean_squared_error(y_test, y_pred_ridge_1)
r2_ridge_1 = r2_score(y_test, y_pred_ridge_1)

print("MAE  :", mae_ridge_1)
print("RMSE :", rmse_ridge_1)
print("R²   :", r2_ridge_1)

MAE  : 1831.6338567194596
RMSE : 3777.2764656343293
R²   : 0.7659892728649353


In [42]:
ridge_10 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=10))
])

ridge_10.fit(X_train, y_train)

y_pred_ridge_10 = ridge_10.predict(X_test)

In [43]:
mae_ridge_10 = mean_absolute_error(y_test, y_pred_ridge_10)
rmse_ridge_10 = root_mean_squared_error(y_test, y_pred_ridge_10)
r2_ridge_10 = r2_score(y_test, y_pred_ridge_10)

print("MAE  :", mae_ridge_10)
print("RMSE :", rmse_ridge_10)
print("R²   :", r2_ridge_10)

MAE  : 2137.013958680844
RMSE : 4088.482753088003
R²   : 0.7258409663504072


### Comparaison des variantes Ridge

Ridge avec `alpha=1` obtient de meilleures performances que la régression linéaire classique (MAE d'environ 1 832 $ et R² d'environ 0,77 contre 2 274 $ et 0,70).

Ces deux modèles n'utilisent toutefois pas le même prétraitement : Ridge utilise des variables numériques standardisées, ce qui n'est pas le cas de la régression linéaire. L'écart observé ne peut donc pas être attribué uniquement à la régularisation.

Avec `alpha=10`, les performances diminuent (MAE d'environ 2 137 $, R² d'environ 0,73). Une régularisation trop forte peut donc limiter excessivement le modèle.

Dans cette expérimentation, `alpha=1` donne le meilleur résultat parmi les variantes Ridge testées.

## RidgeCV

Les expériences précédentes ont testé manuellement plusieurs valeurs de `alpha` pour Ridge.

`RidgeCV` permet de comparer automatiquement plusieurs valeurs de `alpha` à l'aide de la validation croisée et de sélectionner celle qui offre les meilleures performances.

Les valeurs suivantes seront testées :

- `0.01`
- `0.1`
- `1`
- `10`
- `100`

Cette approche permet d'éviter de choisir manuellement le niveau de régularisation.

In [44]:
from sklearn.linear_model import RidgeCV

ridge_cv = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", RidgeCV(
        alphas=[0.01, 0.1, 1, 10, 100],
        cv=5,
        scoring="r2"
    ))
])

ridge_cv.fit(X_train, y_train)

y_pred_ridge_cv = ridge_cv.predict(X_test)

In [45]:
best_alpha_ridge_cv = ridge_cv.named_steps["modele"].alpha_
best_cv_ridge_cv = ridge_cv.named_steps["modele"].best_score_

print("Meilleur alpha :", best_alpha_ridge_cv)
print("R² CV moyen :", best_cv_ridge_cv)

Meilleur alpha : 1.0
R² CV moyen : 0.7844608938201693


In [46]:
mae_ridge_cv = mean_absolute_error(
    y_test,
    y_pred_ridge_cv
)

rmse_ridge_cv = root_mean_squared_error(
    y_test,
    y_pred_ridge_cv
)

r2_ridge_cv = r2_score(
    y_test,
    y_pred_ridge_cv
)

print("MAE  :", mae_ridge_cv)
print("RMSE :", rmse_ridge_cv)
print("R²   :", r2_ridge_cv)

MAE  : 1831.6338567194596
RMSE : 3777.2764656343293
R²   : 0.7659892728649353


### Interprétation de RidgeCV

RidgeCV sélectionne automatiquement `alpha=1` parmi les valeurs testées.

Ce résultat confirme l'expérimentation manuelle précédente, dans laquelle `Ridge(alpha=1)` obtenait les meilleures performances parmi les variantes Ridge.

Les performances sur le jeu de test sont donc identiques à celles de `Ridge(alpha=1)`.

Le score de validation croisée fourni directement par RidgeCV correspond à la validation interne utilisée pour sélectionner `alpha`. Une validation croisée externe est donc effectuée séparément afin de pouvoir comparer RidgeCV aux autres modèles avec le même protocole d'évaluation.

In [47]:
scores_ridge_cv = cross_val_score(
    ridge_cv,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_ridge_cv)
print("R² CV moyen :", scores_ridge_cv.mean())

R² par pli : [0.56466596 0.80967866 0.84448388 0.83083675 0.76818547]
R² CV moyen : 0.7635701434127807


### Conclusion de RidgeCV

La validation croisée externe de RidgeCV obtient un R² moyen d'environ 0,76.

Ce résultat est pratiquement identique à celui obtenu avec `Ridge(alpha=1)`. RidgeCV confirme donc que `alpha=1` constitue un bon niveau de régularisation pour ce jeu de données.

L'intérêt principal de RidgeCV est d'avoir sélectionné automatiquement cette valeur à partir de plusieurs valeurs candidates.

## Lasso

Lasso est une régression linéaire avec régularisation.

Contrairement à Ridge, Lasso peut réduire certains coefficients jusqu'à zéro, ce qui peut éliminer l'influence de certaines variables.

Deux valeurs de `alpha` seront testées :

- `alpha = 1`
- `alpha = 10`

In [48]:
from sklearn.linear_model import Lasso

In [49]:
lasso_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=1,
        max_iter=10000
    ))
])

lasso_1.fit(X_train, y_train)

y_pred_lasso_1 = lasso_1.predict(X_test)

In [50]:
mae_lasso_1 = mean_absolute_error(y_test, y_pred_lasso_1)
rmse_lasso_1 = root_mean_squared_error(y_test, y_pred_lasso_1)
r2_lasso_1 = r2_score(y_test, y_pred_lasso_1)

print("MAE  :", mae_lasso_1)
print("RMSE :", rmse_lasso_1)
print("R²   :", r2_lasso_1)

MAE  : 1725.9744570865007
RMSE : 3847.8011011429658
R²   : 0.7571693794530012


### Interprétation de Lasso alpha=1

Lasso avec `alpha=1` obtient de très bonnes performances.

Sa MAE (environ 1 726 $) est plus faible que celle de la régression linéaire et de Ridge avec `alpha=1`.

Son RMSE (environ 3 848 $) et son R² (environ 0,76) restent toutefois légèrement moins bons que ceux de Ridge avec `alpha=1`.

À ce stade, Lasso `alpha=1` obtient donc la meilleure MAE sur le jeu de test, tandis que Ridge `alpha=1` conserve un RMSE et un R² légèrement meilleurs.

In [51]:
lasso_10 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=10,
        max_iter=10000
    ))
])

lasso_10.fit(X_train, y_train)

y_pred_lasso_10 = lasso_10.predict(X_test)

In [52]:
mae_lasso_10 = mean_absolute_error(y_test, y_pred_lasso_10)
rmse_lasso_10 = root_mean_squared_error(y_test, y_pred_lasso_10)
r2_lasso_10 = r2_score(y_test, y_pred_lasso_10)

print("MAE  :", mae_lasso_10)
print("RMSE :", rmse_lasso_10)
print("R²   :", r2_lasso_10)

MAE  : 2136.4679768959318
RMSE : 4188.242187393228
R²   : 0.7122987196579198


### Comparaison des variantes Lasso

Les deux variantes Lasso obtiennent de bonnes performances.

`alpha=1` présente toutefois une erreur moyenne plus faible que `alpha=10`.

Une régularisation plus forte n'améliore donc pas les performances dans cette expérimentation.

## ElasticNet

ElasticNet combine les régularisations utilisées par Ridge et Lasso.

Le paramètre `alpha` contrôle l'intensité globale de la régularisation, tandis que `l1_ratio` détermine la proportion de régularisation L1 et L2.

Une première expérimentation est réalisée avec :

- `alpha = 1`
- `l1_ratio = 0.5`

Cette configuration donne un poids égal aux composantes Ridge et Lasso.

In [53]:
from sklearn.linear_model import ElasticNet

elasticnet_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", ElasticNet(
        alpha=1,
        l1_ratio=0.5,
        max_iter=10000
    ))
])

elasticnet_1.fit(X_train, y_train)

y_pred_elasticnet_1 = elasticnet_1.predict(X_test)

In [54]:
mae_elasticnet_1 = mean_absolute_error(
    y_test,
    y_pred_elasticnet_1
)

rmse_elasticnet_1 = root_mean_squared_error(
    y_test,
    y_pred_elasticnet_1
)

r2_elasticnet_1 = r2_score(
    y_test,
    y_pred_elasticnet_1
)

print("MAE  :", mae_elasticnet_1)
print("RMSE :", rmse_elasticnet_1)
print("R²   :", r2_elasticnet_1)

MAE  : 3424.398816380819
RMSE : 5556.941873597608
R²   : 0.4935344281198272


### Première interprétation d'ElasticNet

La première configuration d'ElasticNet obtient des performances nettement inférieures à Ridge et Lasso.

Avec `alpha=1` et `l1_ratio=0.5`, la régularisation semble trop importante pour ce jeu de données.

Comme ElasticNet dépend à la fois de `alpha` et de `l1_ratio`, une sélection automatique de ces paramètres par validation croisée est effectuée avant de conclure sur les performances du modèle.

In [55]:
from sklearn.linear_model import ElasticNetCV

elasticnet_cv = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", ElasticNetCV(
        alphas=[0.001, 0.01, 0.1, 1, 10],
        l1_ratio=[0.1, 0.5, 0.9],
        cv=5,
        max_iter=10000
    ))
])

elasticnet_cv.fit(X_train, y_train)

y_pred_elasticnet_cv = elasticnet_cv.predict(X_test)

In [56]:
best_alpha_elasticnet = elasticnet_cv.named_steps["modele"].alpha_
best_l1_ratio_elasticnet = elasticnet_cv.named_steps["modele"].l1_ratio_

print("Meilleur alpha :", best_alpha_elasticnet)
print("Meilleur l1_ratio :", best_l1_ratio_elasticnet)

Meilleur alpha : 0.001
Meilleur l1_ratio : 0.5


In [57]:
mae_elasticnet_cv = mean_absolute_error(
    y_test,
    y_pred_elasticnet_cv
)

rmse_elasticnet_cv = root_mean_squared_error(
    y_test,
    y_pred_elasticnet_cv
)

r2_elasticnet_cv = r2_score(
    y_test,
    y_pred_elasticnet_cv
)

print("MAE  :", mae_elasticnet_cv)
print("RMSE :", rmse_elasticnet_cv)
print("R²   :", r2_elasticnet_cv)

MAE  : 1800.094729207758
RMSE : 3751.0556850165513
R²   : 0.7692268685261618


### Interprétation d'ElasticNetCV

La sélection automatique des hyperparamètres améliore fortement les performances d'ElasticNet par rapport à la configuration initiale.

ElasticNetCV obtient des performances très proches de Ridge avec `alpha=1` : une MAE d'environ 1 800 $ (contre 1 832 $), un RMSE d'environ 3 751 $ (contre 3 777 $) et un R² d'environ 0,769 (contre 0,766).

ElasticNetCV est donc légèrement meilleur sur les trois métriques du jeu de test. Les écarts étant très faibles, aucune conclusion définitive ne peut être tirée à partir du seul jeu de test.

Une validation croisée externe sera utilisée pour comparer plus équitablement les deux approches.

### Remarque sur la grille d'ElasticNetCV

ElasticNetCV sélectionne `alpha=0.001` et `l1_ratio=0.5`.

La valeur `alpha=0.001` correspond à la plus petite valeur proposée dans la grille testée. Comme le meilleur paramètre se trouve à la limite de la grille, une expérimentation avec des valeurs encore plus faibles permettrait de vérifier si une régularisation plus faible serait préférable.

Cet affinement n'a pas été réalisé dans la version actuelle du notebook.

Il faut également noter que l'échelle de `alpha` n'est pas la même pour ElasticNet et pour Ridge dans scikit-learn : les deux valeurs ne peuvent donc pas être comparées directement.

In [58]:
scores_elasticnet_cv = cross_val_score(
    elasticnet_cv,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_elasticnet_cv)
print("R² CV moyen :", scores_elasticnet_cv.mean())

R² par pli : [0.56611177 0.8163044  0.86785081 0.83788993 0.7698156 ]
R² CV moyen : 0.7715945036199483


### Conclusion d'ElasticNetCV

ElasticNetCV obtient un R² moyen d'environ 0,77 en validation croisée externe (environ 0,772).

Ses performances sont très proches de celles des modèles Ridge : RidgeCV obtient un R² moyen d'environ 0,764 et Ridge `alpha=1` d'environ 0,770.

ElasticNetCV obtient également une MAE, un RMSE et un R² légèrement meilleurs que RidgeCV sur le jeu de test.

Les écarts étant très faibles et inférieurs à la variabilité observée entre les plis, les deux modèles sont conservés comme candidats pour la comparaison finale.

## DecisionTreeRegressor

Un arbre de décision est maintenant testé comme modèle de régression.

Contrairement aux modèles linéaires, un arbre peut apprendre des relations non linéaires et des règles de décision successives.

Un arbre sans limitation de profondeur est d'abord utilisé comme référence. Ses performances sur les données d'entraînement et de test permettront également d'observer un éventuel phénomène d'overfitting.

In [59]:
from sklearn.tree import DecisionTreeRegressor

decision_tree = Pipeline([
    ("preparation", preprocessor),
    ("modele", DecisionTreeRegressor(
        random_state=42
    ))
])

decision_tree.fit(X_train, y_train)

y_pred_tree = decision_tree.predict(X_test)
y_pred_tree_train = decision_tree.predict(X_train)

In [60]:
mae_tree = mean_absolute_error(
    y_test,
    y_pred_tree
)

rmse_tree = root_mean_squared_error(
    y_test,
    y_pred_tree
)

r2_tree = r2_score(
    y_test,
    y_pred_tree
)

r2_tree_train = r2_score(
    y_train,
    y_pred_tree_train
)

print("MAE        :", mae_tree)
print("RMSE       :", rmse_tree)
print("R² train   :", r2_tree_train)
print("R² test    :", r2_tree)

MAE        : 2396.8571428571427
RMSE       : 5469.998694175869
R² train   : 1.0
R² test    : 0.5092586358363751


### Interprétation du DecisionTreeRegressor

L'arbre de décision non limité obtient un R² de 1,0 sur les données d'entraînement, mais seulement environ 0,51 sur le jeu de test.

Cet écart important indique un phénomène d'overfitting : l'arbre apprend très précisément les données d'entraînement, mais généralise moins bien sur de nouvelles observations.

Une limitation de la profondeur de l'arbre est donc testée afin de réduire sa complexité et d'améliorer sa capacité de généralisation.

In [61]:
from sklearn.tree import DecisionTreeRegressor

profondeurs = [3, 5, 10, None]

resultats_tree = []

for profondeur in profondeurs:

    tree = Pipeline([
        ("preparation", preprocessor),
        ("modele", DecisionTreeRegressor(
            max_depth=profondeur,
            random_state=42
        ))
    ])

    tree.fit(X_train, y_train)

    pred_train = tree.predict(X_train)
    pred_test = tree.predict(X_test)

    resultats_tree.append({
        "max_depth": profondeur,
        "R² train": r2_score(y_train, pred_train),
        "R² test": r2_score(y_test, pred_test)
    })

pd.DataFrame(resultats_tree)

,max_depth,R² train,R² test
0,3.0,0.562613,0.270023
1,5.0,0.688490,0.335284
2,10.0,0.897598,0.489608
3,NaN,1.000000,0.509259


### Régularisation avec min_samples_leaf

La limitation de la profondeur n'a pas permis d'améliorer les performances du DecisionTreeRegressor sur le jeu de test.

On teste maintenant `min_samples_leaf`, qui impose un nombre minimal d'observations dans chaque feuille de l'arbre.

Cette contrainte peut réduire l'overfitting sans limiter directement la profondeur maximale de l'arbre.

In [62]:
min_leaf_values = [1, 2, 5, 10, 20]

resultats_tree_leaf = []

for min_leaf in min_leaf_values:

    tree = Pipeline([
        ("preparation", preprocessor),
        ("modele", DecisionTreeRegressor(
            min_samples_leaf=min_leaf,
            random_state=42
        ))
    ])

    tree.fit(X_train, y_train)

    pred_train = tree.predict(X_train)
    pred_test = tree.predict(X_test)

    resultats_tree_leaf.append({
        "min_samples_leaf": min_leaf,
        "R² train": r2_score(y_train, pred_train),
        "R² test": r2_score(y_test, pred_test)
    })

pd.DataFrame(resultats_tree_leaf)

,min_samples_leaf,R² train,R² test
0,1,1.000000,0.509259
1,2,0.948945,0.644947
2,5,0.796395,0.589010
3,10,0.724444,0.533125
4,20,0.659894,0.546047


### Conclusion du DecisionTreeRegressor

Le DecisionTreeRegressor non régularisé présente un fort overfitting avec un R² de 1,0 sur les données d'entraînement et environ 0,51 sur le jeu de test.

La limitation de `max_depth` n'a pas amélioré les performances sur le jeu de test (R² test entre 0,27 et 0,49).

L'utilisation de `min_samples_leaf` permet toutefois de réduire l'overfitting. Parmi les valeurs testées, `min_samples_leaf=2` obtient le meilleur R² test, soit environ 0,64 (R² entraînement d'environ 0,95).

Cette variante améliore donc la généralisation de l'arbre, mais ses performances restent inférieures à celles des modèles Ridge et ElasticNet.

Ces réglages ont été choisis en comparant les résultats sur le jeu de test, qui joue ici le rôle de jeu de validation.

In [63]:
decision_tree_best = Pipeline([
    ("preparation", preprocessor),
    ("modele", DecisionTreeRegressor(
        min_samples_leaf=2,
        random_state=42
    ))
])

decision_tree_best.fit(X_train, y_train)

y_pred_tree_best = decision_tree_best.predict(X_test)

mae_tree_best = mean_absolute_error(
    y_test,
    y_pred_tree_best
)

rmse_tree_best = root_mean_squared_error(
    y_test,
    y_pred_tree_best
)

r2_tree_best = r2_score(
    y_test,
    y_pred_tree_best
)

print("MAE  :", mae_tree_best)
print("RMSE :", rmse_tree_best)
print("R²   :", r2_tree_best)

MAE  : 2581.5714285714284
RMSE : 4652.7277372338685
R²   : 0.6449467437252341


In [64]:
scores_tree_best = cross_val_score(
    decision_tree_best,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_tree_best)
print("R² CV moyen :", scores_tree_best.mean())

R² par pli : [0.4766337  0.4296074  0.54168951 0.73295456 0.09270674]
R² CV moyen : 0.4547183829269863


In [65]:
DecisionTreeRegressor(
    min_samples_leaf=2,
    random_state=42
)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [66]:
decision_tree_best = Pipeline([
    ("preparation", preprocessor),
    ("modele", DecisionTreeRegressor(
        min_samples_leaf=2,
        random_state=42
    ))
])

decision_tree_best.fit(X_train, y_train)

y_pred_tree_best = decision_tree_best.predict(X_test)

mae_tree_best = mean_absolute_error(
    y_test,
    y_pred_tree_best
)

rmse_tree_best = root_mean_squared_error(
    y_test,
    y_pred_tree_best
)

r2_tree_best = r2_score(
    y_test,
    y_pred_tree_best
)

print("MAE  :", mae_tree_best)
print("RMSE :", rmse_tree_best)
print("R²   :", r2_tree_best)

MAE  : 2581.5714285714284
RMSE : 4652.7277372338685
R²   : 0.6449467437252341


In [67]:
scores_tree_best = cross_val_score(
    decision_tree_best,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_tree_best)
print("R² CV moyen :", scores_tree_best.mean())

R² par pli : [0.4766337  0.4296074  0.54168951 0.73295456 0.09270674]
R² CV moyen : 0.4547183829269863


### Conclusion de la validation croisée du DecisionTreeRegressor

Le DecisionTreeRegressor avec `min_samples_leaf=2` obtient un R² moyen d'environ 0,45 en validation croisée.

Les scores varient fortement d'un pli à l'autre (d'environ 0,09 à 0,73), ce qui indique que les performances de l'arbre dépendent beaucoup des données utilisées pour l'entraînement.

La régularisation a permis de réduire l'overfitting observé avec l'arbre non limité, mais le modèle reste moins performant et moins stable que RidgeCV et ElasticNetCV.

Le DecisionTreeRegressor n'est donc pas retenu comme candidat principal à ce stade.

## Variantes du RandomForest

In [68]:
random_forest_200 = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

random_forest_200.fit(X_train_prepared, y_train)

y_pred_forest_200 = random_forest_200.predict(X_test_prepared)

In [69]:
mae_forest_200 = mean_absolute_error(y_test, y_pred_forest_200)
rmse_forest_200 = root_mean_squared_error(y_test, y_pred_forest_200)
r2_forest_200 = r2_score(y_test, y_pred_forest_200)

print("MAE  :", mae_forest_200)
print("RMSE :", rmse_forest_200)
print("R²   :", r2_forest_200)

MAE  : 1962.4864285714286
RMSE : 4585.0557407618135
R²   : 0.6551998388950869


### Effet du nombre d'arbres

Le passage de 100 à 200 arbres ne modifie pratiquement pas les performances du RandomForestRegressor.

La MAE diminue très légèrement (environ 1 962 $ contre 1 973 $), tandis que le RMSE augmente légèrement (environ 4 585 $ contre 4 537 $) et que le R² diminue légèrement (environ 0,655 contre 0,662).

Augmenter le nombre d'arbres n'apporte donc pas d'amélioration dans cette expérimentation.

In [70]:
random_forest_depth_10 = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

random_forest_depth_10.fit(X_train_prepared, y_train)

y_pred_forest_depth_10 = random_forest_depth_10.predict(X_test_prepared)

In [71]:
mae_forest_depth_10 = mean_absolute_error(
    y_test,
    y_pred_forest_depth_10
)

rmse_forest_depth_10 = root_mean_squared_error(
    y_test,
    y_pred_forest_depth_10
)

r2_forest_depth_10 = r2_score(
    y_test,
    y_pred_forest_depth_10
)

print("MAE  :", mae_forest_depth_10)
print("RMSE :", rmse_forest_depth_10)
print("R²   :", r2_forest_depth_10)

MAE  : 2298.477030119256
RMSE : 4802.083161424246
R²   : 0.6217860231347151


### Effet de la profondeur des arbres

Limiter la profondeur des arbres à 10 n'améliore pas les performances du RandomForestRegressor.

Les erreurs augmentent (MAE d'environ 2 298 $ contre 1 973 $, RMSE d'environ 4 802 $ contre 4 537 $) et le R² diminue (environ 0,62 contre 0,66). Dans cette expérimentation, la limitation de profondeur n'apporte donc pas de bénéfice.

## XGBoost

XGBoost est une méthode d'ensemble basée sur le boosting.

Contrairement au RandomForest, où plusieurs arbres sont construits de manière relativement indépendante, XGBoost construit les arbres de manière séquentielle.

Chaque nouvel arbre cherche à corriger une partie des erreurs commises par les arbres précédents.

Une première configuration simple est utilisée avant toute optimisation des hyperparamètres.

In [72]:
from xgboost import XGBRegressor

xgb_model = Pipeline([
    ("preparation", preprocessor),
    ("modele", XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        objective="reg:squarederror"
    ))
])

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

In [73]:
mae_xgb = mean_absolute_error(
    y_test,
    y_pred_xgb
)

rmse_xgb = root_mean_squared_error(
    y_test,
    y_pred_xgb
)

r2_xgb = r2_score(
    y_test,
    y_pred_xgb
)

print("MAE  :", mae_xgb)
print("RMSE :", rmse_xgb)
print("R²   :", r2_xgb)

MAE  : 2109.024169921875
RMSE : 4502.5009765625
R²   : 0.6675044298171997


### Première interprétation de XGBoost

La première configuration de XGBoost obtient un R² d'environ 0,67 sur le jeu de test, avec une MAE d'environ 2 109 $ et un RMSE d'environ 4 503 $.

Sur ce jeu de test, ces résultats sont inférieurs à ceux des modèles linéaires régularisés (Ridge, Lasso et ElasticNetCV), qui obtiennent un R² d'environ 0,76 à 0,77 et un RMSE d'environ 3 751 $ à 3 848 $.

XGBoost fait toutefois mieux que l'arbre de décision seul et obtient un R² test légèrement supérieur au RandomForestRegressor.

Une validation croisée est nécessaire avant de conclure sur sa capacité de généralisation.

In [74]:
y_pred_xgb_train = xgb_model.predict(X_train)

r2_xgb_train = r2_score(
    y_train,
    y_pred_xgb_train
)

print("R² train :", r2_xgb_train)
print("R² test  :", r2_xgb)

R² train : 0.904416561126709
R² test  : 0.6675044298171997


In [75]:
scores_xgb = cross_val_score(
    xgb_model,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_xgb)
print("R² CV moyen :", scores_xgb.mean())

R² par pli : [0.75561637 0.82446158 0.79738486 0.79529011 0.67076451]
R² CV moyen : 0.7687034845352173


### Ajustement léger des hyperparamètres de XGBoost

La première configuration de XGBoost présente de bonnes performances, mais un écart reste visible entre le R² d'entraînement et le R² de test.

Quelques configurations sont donc testées en faisant varier :

- le nombre d'arbres (`n_estimators`) ;
- le taux d'apprentissage (`learning_rate`) ;
- la profondeur maximale des arbres (`max_depth`).

L'objectif n'est pas d'effectuer une recherche exhaustive, mais d'identifier une configuration offrant un meilleur compromis entre performance et généralisation.

In [76]:
configurations_xgb = [
    {
        "nom": "baseline",
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 3
    },
    {
        "nom": "200 arbres - lr 0.05 - depth 3",
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 3
    },
    {
        "nom": "300 arbres - lr 0.03 - depth 3",
        "n_estimators": 300,
        "learning_rate": 0.03,
        "max_depth": 3
    },
    {
        "nom": "200 arbres - lr 0.05 - depth 2",
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 2
    },
    {
        "nom": "300 arbres - lr 0.03 - depth 2",
        "n_estimators": 300,
        "learning_rate": 0.03,
        "max_depth": 2
    },
    {
        "nom": "200 arbres - lr 0.05 - depth 4",
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 4
    }
]

resultats_xgb_tuning = []

for config in configurations_xgb:

    modele = Pipeline([
        ("preparation", preprocessor),
        ("modele", XGBRegressor(
            n_estimators=config["n_estimators"],
            learning_rate=config["learning_rate"],
            max_depth=config["max_depth"],
            random_state=42,
            objective="reg:squarederror"
        ))
    ])

    modele.fit(X_train, y_train)

    pred_train = modele.predict(X_train)
    pred_test = modele.predict(X_test)

    resultats_xgb_tuning.append({
        "Configuration": config["nom"],
        "R² train": r2_score(y_train, pred_train),
        "R² test": r2_score(y_test, pred_test),
        "MAE": mean_absolute_error(y_test, pred_test),
        "RMSE": root_mean_squared_error(y_test, pred_test)
    })

resultats_xgb_tuning = pd.DataFrame(resultats_xgb_tuning)

resultats_xgb_tuning

,Configuration,R² train,R² test,MAE,RMSE
0,baseline,0.904417,0.667504,2109.024170,4502.500977
1,200 arbres - lr 0.05 - depth 3,0.904013,0.666755,2083.352051,4507.569336
2,300 arbres - lr 0.03 - depth 3,0.896585,0.658725,2138.181396,4561.559570
3,200 arbres - lr 0.05 - depth 2,0.852133,0.641522,2363.691162,4675.114258
4,300 arbres - lr 0.03 - depth 2,0.843041,0.635908,2417.200684,4711.576660
5,200 arbres - lr 0.05 - depth 4,0.929023,0.632629,2065.022217,4732.748535


### Conclusion de l'ajustement de XGBoost

Les différentes configurations testées ne permettent pas d'améliorer clairement la configuration initiale de XGBoost.

La configuration de référence avec `100` arbres, un `learning_rate` de `0.1` et une profondeur maximale de `3` obtient le meilleur R² sur le jeu de test ainsi que le meilleur RMSE parmi les variantes testées.

Des arbres moins profonds réduisent légèrement l'écart entre les performances d'entraînement et de test, mais entraînent également une diminution des performances prédictives.

Une profondeur de `4` et la configuration à 200 arbres avec `learning_rate=0.05` réduisent légèrement le MAE, mais dégradent le RMSE et le R² test ; la profondeur de `4` augmente aussi l'overfitting.

La configuration initiale est donc conservée comme variante XGBoost de référence pour la comparaison finale des modèles.

### Interprétation de la validation croisée de XGBoost

XGBoost obtient un R² moyen d'environ 0,77 en validation croisée à 5 plis (environ 0,769), très proche d'ElasticNetCV (environ 0,772).

Ses scores sont plus homogènes entre les plis (environ 0,67 à 0,82) que ceux des modèles linéaires, notamment sur le premier pli.

En revanche, son R² sur le jeu de test (environ 0,67) est nettement inférieur à sa moyenne en validation croisée : la séparation train/test utilisée lui est moins favorable qu'aux modèles linéaires.

XGBoost reste donc un candidat important pour la comparaison finale, mais ses résultats dépendent davantage du découpage utilisé.

## Validation croisée de Ridge

La validation croisée à 5 plis permet de vérifier si les bonnes performances de Ridge avec `alpha=1` restent stables sur plusieurs séparations des données.

In [77]:
pipeline_ridge_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=1))
])

In [78]:
scores_ridge_1 = cross_val_score(
    pipeline_ridge_1,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_ridge_1)
print("R² moyen   :", scores_ridge_1.mean())

R² par pli : [0.56466596 0.80967866 0.86784154 0.83702059 0.76818547]
R² moyen   : 0.7694784412888979


### Interprétation de la validation croisée de Ridge

Ridge avec `alpha=1` obtient un R² moyen d'environ 0,77 sur les 5 plis.

Ce résultat est supérieur à celui de la régression linéaire classique (environ 0,72). Comme Ridge utilise des variables numériques standardisées, contrairement à la régression linéaire, cet écart combine l'effet de la régularisation et celui de la standardisation.

In [79]:
pipeline_lasso_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=1,
        max_iter=10000
    ))
])

In [80]:
scores_lasso_1 = cross_val_score(
    pipeline_lasso_1,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_lasso_1)
print("R² moyen   :", scores_lasso_1.mean())

R² par pli : [0.56284637 0.84067685 0.78732995 0.82367361 0.75976077]
R² moyen   : 0.7548575098928625


### Interprétation de la validation croisée de Lasso

Lasso avec `alpha=1` obtient un R² moyen d'environ 0,76 sur les 5 plis.

Ce résultat confirme les bonnes performances du modèle. Cependant, Ridge avec `alpha=1` obtient un R² moyen légèrement supérieur, autour de 0,77.

Les deux modèles régularisés améliorent les performances par rapport à la régression linéaire classique.

## Comparaison finale des expériences

Les différentes variantes sont regroupées afin de comparer leurs performances sur le même jeu de test.

La validation croisée est présentée pour les principaux modèles retenus.

In [81]:
resultats_finaux = pd.DataFrame({
    "Modèle": [
        "Dummy - Mean",
        "Dummy - Median",
        "LinearRegression",
        "Ridge alpha=1",
        "RidgeCV",
        "Lasso alpha=1",
        "ElasticNetCV",
        "DecisionTree - min_samples_leaf=2",
        "RandomForest - 100 arbres",
        "XGBoost"
    ],

    "MAE": [
        mae_dummy,
        mae_dummy_median,
        mae_linear,
        mae_ridge_1,
        mae_ridge_cv,
        mae_lasso_1,
        mae_elasticnet_cv,
        mae_tree_best,
        mae_forest,
        mae_xgb
    ],

    "RMSE": [
        rmse_dummy,
        rmse_dummy_median,
        rmse_linear,
        rmse_ridge_1,
        rmse_ridge_cv,
        rmse_lasso_1,
        rmse_elasticnet_cv,
        rmse_tree_best,
        rmse_forest,
        rmse_xgb
    ],

    "R² test": [
        r2_dummy,
        r2_dummy_median,
        r2_linear,
        r2_ridge_1,
        r2_ridge_cv,
        r2_lasso_1,
        r2_elasticnet_cv,
        r2_tree_best,
        r2_forest,
        r2_xgb
    ],

    "R² CV moyen": [
        np.nan,
        np.nan,
        scores_linear.mean(),
        scores_ridge_1.mean(),
        scores_ridge_cv.mean(),
        scores_lasso_1.mean(),
        scores_elasticnet_cv.mean(),
        scores_tree_best.mean(),
        scores_forest.mean(),
        scores_xgb.mean()
    ]
})

resultats_finaux.round(3)

,Modèle,MAE,RMSE,R² test,R² CV moyen
0,Dummy - Mean,5740.730,7872.427,-0.016,NaN
1,Dummy - Median,5543.429,7814.295,-0.002,NaN
2,LinearRegression,2274.190,4242.585,0.705,0.723
3,Ridge alpha=1,1831.634,3777.276,0.766,0.769
4,RidgeCV,1831.634,3777.276,0.766,0.764
5,Lasso alpha=1,1725.974,3847.801,0.757,0.755
6,ElasticNetCV,1800.095,3751.056,0.769,0.772
7,DecisionTree - min_samples_leaf=2,2581.571,4652.728,0.645,0.455
8,RandomForest - 100 arbres,1972.823,4537.213,0.662,0.711
9,XGBoost,2109.024,4502.501,0.668,0.769


### Analyse finale des modèles

Les modèles testés obtiennent des performances très supérieures aux modèles Dummy, ce qui confirme que les variables sélectionnées permettent de prédire une part importante du prix des véhicules.

`ElasticNetCV` présente le meilleur compromis global. Il obtient le meilleur RMSE et le meilleur R² sur le jeu de test, ainsi que le R² moyen le plus élevé en validation croisée (environ 0,772).

`Lasso(alpha=1)` obtient la meilleure MAE, ce qui signifie qu'il produit en moyenne les erreurs absolues les plus faibles sur le jeu de test.

`Ridge(alpha=1)` et `RidgeCV` présentent également de très bonnes performances et une bonne cohérence entre le jeu de test et la validation croisée.

XGBoost obtient un R² moyen en validation croisée très proche de celui d'ElasticNetCV (environ 0,769) et des scores plus homogènes entre les plis. Ses performances sur le jeu de test sont toutefois inférieures à celles des modèles régularisés, et l'écart entre ses performances d'entraînement (R² d'environ 0,90) et de test indique une tendance plus importante à l'overfitting.

Le RandomForest et le DecisionTreeRegressor présentent des performances inférieures aux meilleurs modèles. Le DecisionTree est notamment très instable en validation croisée.

Les écarts entre les modèles linéaires régularisés restent très faibles (moins de 0,02 de R² moyen en validation croisée) et inférieurs à la variabilité observée entre les plis. Le choix du champion repose donc sur un compromis et non sur une supériorité démontrée.

Le jeu de test ayant servi à comparer les modèles et à choisir le champion, son score constitue une estimation légèrement optimiste des performances.

À ce stade, ElasticNetCV est retenu comme modèle champion, tandis que Lasso, Ridge et XGBoost sont conservés comme principaux modèles de comparaison.

# Conclusion

L'objectif de ce TP était de prédire le prix d'un véhicule à partir de plusieurs caractéristiques comme l'année, le kilométrage, la marque, le modèle, le type de carrosserie, le carburant et la transmission.

Un premier modèle de référence avec `DummyRegressor` a permis d'établir une performance minimale. Ses résultats montrent qu'une prédiction basée uniquement sur la moyenne ou la médiane des prix produit une erreur importante (MAE d'environ 5 500 $ à 5 700 $) et un R² proche de 0.

La `LinearRegression` améliore fortement les résultats (MAE d'environ 2 274 $, R² test d'environ 0,70) et montre qu'une partie importante du prix peut être expliquée par une relation linéaire entre les caractéristiques des véhicules et leur prix.

Les modèles linéaires régularisés, utilisés avec des variables numériques standardisées, donnent les meilleurs résultats sur le jeu de test. `ElasticNetCV` obtient le meilleur RMSE (environ 3 751 $) et le meilleur R² test (environ 0,769), tandis que `Lasso(alpha=1)` obtient la MAE la plus faible (environ 1 726 $).

La validation croisée apporte une information supplémentaire. Les R² moyens des modèles régularisés sont très proches (environ 0,755 à 0,772), et XGBoost obtient un R² moyen presque identique à celui d'ElasticNetCV (environ 0,769) malgré des performances plus faibles sur le jeu de test. Cela montre qu'il est important de ne pas évaluer un modèle uniquement sur une seule séparation train/test.

Le `DecisionTreeRegressor` présente un fort overfitting et une grande instabilité. Le `RandomForestRegressor` reste inférieur aux modèles régularisés : l'augmentation du nombre d'arbres de 100 à 200 n'apporte pas d'amélioration, et limiter la profondeur des arbres à 10 réduit les performances.

En conclusion, `ElasticNetCV` est retenu comme modèle champion pour son meilleur compromis entre MAE, RMSE, R² test et R² moyen en validation croisée. Les écarts avec Ridge et Lasso restent toutefois très faibles. Ces résultats montrent également qu'un modèle plus complexe n'est pas automatiquement plus performant et que plusieurs méthodes d'évaluation sont nécessaires pour comparer correctement les modèles.